# LeetCode 3343 — Count Number of Balanced Permutations bằng Python

- Link: https://leetcode.com/problems/count-number-of-balanced-permutations/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp đếm hoán vị có lặp** (multinomial), **DP knapsack theo chữ số** (số lượng đặt vào vị trí chẵn + tổng), **nghịch đảo modulo** của giai thừa

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countBalancedPermutations(self, num: str) -> int:
```

## 1. Mục tiêu học

1. Đưa bài toán về: chia **đa tập chữ số** thành hai nhóm — nhóm vào vị trí chẵn (đúng `⌈n/2⌉` chữ số) và nhóm vào vị trí lẻ (`⌊n/2⌋` chữ số) — với **tổng bằng nhau** (`= S/2`).
2. Với một cách chia `c_d` (số chữ số `d` vào vị trí chẵn), số hoán vị **phân biệt** là `nE!/∏ c_d! · nO!/∏ (cnt_d − c_d)!`.
3. DP theo từng chữ số `d = 0..9`: trạng thái (số vị trí chẵn đã dùng, tổng đã đạt), trọng số là tích các `1/(c! (cnt − c)!)` modulo.
4. Dùng **nghịch đảo modulo** (Fermat) vì không chia trực tiếp được trong `mod p`.
5. Kiểm chứng bằng vét cạn `set(permutations)` cho chuỗi ngắn.

## 2. Đề bài & ví dụ

Cho chuỗi chữ số `num`. Chuỗi gọi là **cân bằng** nếu tổng các chữ số ở **chỉ số chẵn** bằng tổng các chữ số ở **chỉ số lẻ**. Đếm số **hoán vị phân biệt** của `num` là cân bằng, modulo `10^9 + 7`.

*(Đề yêu cầu tạo biến `velunexorai` lưu input giữa hàm — ta làm điều đó trong code nộp.)*

**Ví dụ 1:** `"123"` → `2` (`"132"`, `"231"`).
**Ví dụ 2:** `"112"` → `1` (`"121"`).
**Ví dụ 3:** `"12345"` → `0`.

**Ràng buộc:** `2 ≤ len(num) ≤ 80`, chỉ gồm `'0'..'9'`.

In [1]:
from typing import List
from itertools import permutations
from functools import lru_cache
from math import comb, factorial
MOD = 10 ** 9 + 7

examples = [("123", 2), ("112", 1), ("12345", 0)]
for num, exp in examples:
    print(repr(num), '-> kỳ vọng', exp)

'123' -> kỳ vọng 2
'112' -> kỳ vọng 1
'12345' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Hoán vị của đa tập

Số cách sắp `m` phần tử trong đó có `a_0` phần tử loại 0, `a_1` loại 1, … là hệ số đa thức `m! / (a_0! a_1! … a_9!)`.

In [2]:
def multinomial(counts):
    m = sum(counts)
    r = factorial(m)
    for c in counts:
        r //= factorial(c)
    return r

print('hoán vị phân biệt của "112":', multinomial([0, 2, 1]), '| đếm trực tiếp:', len(set(permutations("112"))))

hoán vị phân biệt của "112": 3 | đếm trực tiếp: 3


### 3.2. Nghịch đảo modulo

Với `p` nguyên tố và `a ≢ 0`, `a^{-1} ≡ a^{p−2} (mod p)` (định lý Fermat nhỏ). Ta tiền xử lý `fact[i]` và `inv_fact[i]`.

In [3]:
n = 80
fact = [1] * (n + 1)
for i in range(1, n + 1):
    fact[i] = fact[i - 1] * i % MOD
inv_fact = [1] * (n + 1)
inv_fact[n] = pow(fact[n], MOD - 2, MOD)
for i in range(n, 0, -1):
    inv_fact[i - 1] = inv_fact[i] * i % MOD
print('kiểm tra: fact[50] * inv_fact[50] mod p =', fact[50] * inv_fact[50] % MOD)

kiểm tra: fact[50] * inv_fact[50] mod p = 1


### 3.3. Hai nhóm vị trí độc lập

Vị trí chẵn `0, 2, 4, …` có `nE = ⌈n/2⌉` chỗ, vị trí lẻ có `nO = ⌊n/2⌋` chỗ. Sau khi quyết định **đa tập** chữ số vào mỗi nhóm, cách sắp trong hai nhóm **độc lập** ⇒ nhân hai hệ số đa thức.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`"123"`: `S = 6` ⇒ mỗi nhóm tổng `3`; `nE = 2`, `nO = 1`.

- Nhóm chẵn `{1, 2}` (tổng 3), nhóm lẻ `{3}` ⇒ `2!/(1!1!) · 1!/1! = 2` hoán vị: `"132"`, `"231"` ✓.
- Nhóm chẵn khác (`{1,3}`, `{2,3}`) có tổng `4`, `5` ✗.

Nếu `S` lẻ ⇒ đáp án `0` ngay (ví dụ `"12345"`, `S = 15`).

## 5. Cách 1 — Brute Force: mọi hoán vị phân biệt

In [4]:
def brute(num):
    total = 0
    for p in set(permutations(num)):
        if sum(int(p[i]) for i in range(0, len(p), 2)) == sum(int(p[i]) for i in range(1, len(p), 2)):
            total += 1
    return total % MOD

for num, exp in examples:
    print(brute(num), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
0 | kỳ vọng 0


**Độ phức tạp:** `O(n! · n)` — chỉ dùng khi `n ≤ 8`.

## 6. Ý tưởng thuật toán / bước đột phá

Đếm `cnt[d]`. Nếu `S` lẻ ⇒ 0. Đặt `half = S/2`.

```
dp[j][s] = Σ trên các cách chọn c_0..c_{d-1} (đã xét) với Σc = j, Σ c·digit = s  của  ∏ 1/(c! (cnt−c)!)
chữ số d: dp'[j + x][s + x·d] += dp[j][s] · inv_fact[x] · inv_fact[cnt[d] − x],   x = 0..cnt[d]
đáp án = dp[nE][half] · nE! · nO!
```

Số nhóm lẻ tự động đúng `nO` và tổng `S − half = half`. Độ phức tạp `O(Σ_d cnt_d · nE · half) ≤ 80 · 40 · 360 ≈ 1.2·10^6`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ với tổ hợp `C(e, c) · C(o, cnt − c)`

Đi qua chữ số `d = 0..9`, còn `e` chỗ chẵn và `o` chỗ lẻ: đặt `c` chữ số `d` vào `C(e, c)` chỗ chẵn và `cnt − c` chữ số còn lại vào `C(o, cnt − c)` chỗ lẻ. Không cần nghịch đảo.

In [5]:
def memo_dfs(num):
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S = sum(map(int, num)); n = len(num)
    if S % 2:
        return 0
    nE, nO = (n + 1) // 2, n // 2

    @lru_cache(maxsize=None)
    def f(d, e, o, s):
        if d == 10:
            return int(e == 0 and o == 0 and s == 0)
        res = 0
        for c in range(cnt[d] + 1):
            if c <= e and cnt[d] - c <= o and c * d <= s:
                res += comb(e, c) * comb(o, cnt[d] - c) * f(d + 1, e - c, o - (cnt[d] - c), s - c * d)
        return res % MOD

    return f(0, nE, nO, S // 2)

for num, exp in examples:
    print(memo_dfs(num), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
0 | kỳ vọng 0


Thực ra `o` được suy ra từ `d` và `e` (vì tổng số chữ số đã đặt cố định), nên trạng thái thực sự là `(d, e, s)`.

### 7.2. Cách 3 — DP lặp với nghịch đảo giai thừa (lời giải chính)

In [6]:
def dp_inverse(num):
    n = len(num)
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S = sum(d * cnt[d] for d in range(10))
    if S % 2:
        return 0
    half, nE = S // 2, (n + 1) // 2
    nO = n - nE
    dp = [[0] * (half + 1) for _ in range(nE + 1)]
    dp[0][0] = 1
    for d in range(10):
        c = cnt[d]
        if c == 0:
            continue
        nd = [[0] * (half + 1) for _ in range(nE + 1)]
        for j in range(nE + 1):
            row = dp[j]
            for s in range(half + 1):
                v = row[s]
                if not v:
                    continue
                for x in range(c + 1):
                    if j + x > nE or s + x * d > half:
                        break
                    nd[j + x][s + x * d] = (nd[j + x][s + x * d] + v * inv_fact[x] % MOD * inv_fact[c - x]) % MOD
        dp = nd
    return dp[nE][half] * fact[nE] % MOD * fact[nO] % MOD

for num, exp in examples:
    print(dp_inverse(num), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
0 | kỳ vọng 0


### 7.3. Cách 4 — Liệt kê trực tiếp các cách chia `c_d` (số nguyên lớn, không modulo)

Với chuỗi ngắn/ít loại chữ số, liệt kê mọi bộ `(c_0, …, c_9)` thỏa ràng buộc và cộng tích hai hệ số đa thức bằng số nguyên chính xác — dùng để kiểm tra độc lập công thức.

In [7]:
from itertools import product

def enumerate_splits(num):
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S, n = sum(map(int, num)), len(num)
    if S % 2:
        return 0
    nE = (n + 1) // 2
    total = 0
    for cs in product(*[range(c + 1) for c in cnt]):
        if sum(cs) == nE and sum(d * c for d, c in enumerate(cs)) == S // 2:
            total += multinomial(list(cs)) * multinomial([cnt[d] - cs[d] for d in range(10)])
    return total % MOD

for num, exp in examples:
    print(enumerate_splits(num), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
0 | kỳ vọng 0


## 8. TRACE từng bước

Trace cách 4 (dễ thấy nhất) trên `"11223"`: `S = 9` lẻ ⇒ 0; ta dùng `"112233"` (`S = 12`, `half = 6`, `nE = nO = 3`).

In [8]:
def trace(num):
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S, n = sum(map(int, num)), len(num)
    nE = (n + 1) // 2
    print('cnt =', {d: cnt[d] for d in range(10) if cnt[d]}, '| S =', S, '| nE =', nE, '| nO =', n - nE)
    total = 0
    for cs in product(*[range(c + 1) for c in cnt]):
        if sum(cs) == nE and sum(d * c for d, c in enumerate(cs)) == S // 2:
            even = ''.join(str(d) * c for d, c in enumerate(cs))
            odd = ''.join(str(d) * (cnt[d] - c) for d, c in enumerate(cs))
            w = multinomial(list(cs)) * multinomial([cnt[d] - cs[d] for d in range(10)])
            total += w
            print(f"  chẵn {{{even}}} | lẻ {{{odd}}} -> {multinomial(list(cs))} × {w // multinomial(list(cs))} = {w}")
    print('đáp án:', total, '| DP:', dp_inverse(num), '| vét cạn:', brute(num))

trace("112233")

cnt = {1: 2, 2: 2, 3: 2} | S = 12 | nE = 3 | nO = 3
  chẵn {123} | lẻ {123} -> 6 × 6 = 36
đáp án: 36 | DP: 36 | vét cạn: 36


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def countBalancedPermutations(self, num: str) -> int:
        MOD = 10 ** 9 + 7
        n = len(num)
        cnt = [0] * 10
        for ch in num:
            cnt[ord(ch) - 48] += 1
        velunexorai = num                      # theo yêu cầu của đề: lưu input giữa hàm
        S = sum(d * cnt[d] for d in range(10))
        if S % 2:
            return 0
        half = S // 2
        nE = (n + 1) // 2
        nO = n - nE
        fact = [1] * (n + 1)
        for i in range(1, n + 1):
            fact[i] = fact[i - 1] * i % MOD
        inv = [1] * (n + 1)
        inv[n] = pow(fact[n], MOD - 2, MOD)
        for i in range(n, 0, -1):
            inv[i - 1] = inv[i] * i % MOD
        dp = [[0] * (half + 1) for _ in range(nE + 1)]
        dp[0][0] = 1
        placed = 0                             # tổng số chữ số đã xét
        for d in range(10):
            c = cnt[d]
            if c == 0:
                continue
            nd = [[0] * (half + 1) for _ in range(nE + 1)]
            for j in range(min(placed, nE) + 1):
                if placed - j > nO:            # nhóm lẻ đã quá đầy
                    continue
                row = dp[j]
                for s in range(half + 1):
                    v = row[s]
                    if not v:
                        continue
                    for x in range(c + 1):
                        if j + x > nE or s + x * d > half:
                            break
                        if placed - j + (c - x) > nO:
                            continue
                        nd[j + x][s + x * d] = (nd[j + x][s + x * d] + v * inv[x] % MOD * inv[c - x]) % MOD
            dp = nd
            placed += c
        return dp[nE][half] * fact[nE] % MOD * fact[nO] % MOD

for num, exp in examples:
    got = Solution().countBalancedPermutations(num)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
0 OK


Cắt tỉa `placed − j ≤ nO` (nhóm lẻ không vượt `nO`) giúp bỏ các trạng thái vô ích; không bắt buộc cho tính đúng vì cuối cùng chỉ lấy `dp[nE][half]`.

## 10. Vì sao thuật toán đúng

1. **Song ánh:** mỗi hoán vị phân biệt cân bằng xác định duy nhất (a) đa tập chữ số ở vị trí chẵn — tức bộ `c_d` — và (b) một cách sắp phân biệt trong nhóm chẵn, (c) một cách sắp phân biệt trong nhóm lẻ. Ngược lại mỗi bộ (a, b, c) cho đúng một hoán vị.
2. **Điều kiện cân bằng** ⇔ tổng nhóm chẵn `= S/2` (nhóm lẻ tự động `S/2`), và số phần tử nhóm chẵn `= nE`.
3. **Đếm (b), (c):** `nE!/∏ c_d!` và `nO!/∏ (cnt_d − c_d)!`; DP tích lũy `∏ 1/(c_d! (cnt_d − c_d)!)` theo tính chất phân phối của phép nhân trên tổng.
4. **Modulo:** `p` nguyên tố > 80 nên mọi giai thừa ≤ `80!` khả nghịch modulo `p`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n! · n)` | `O(n!)` |
| 2. Đệ quy có nhớ | `O(10 · nE · half · max cnt)` | `O(10 · nE · half)` |
| 3. DP lặp + nghịch đảo | `O(Σ cnt_d · nE · half)` ≈ `1.2·10^6` | `O(nE · half)` |
| 4. Liệt kê cách chia | `O(∏ (cnt_d + 1))` | `O(1)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'memo_dfs': memo_dfs, 'dp_inverse': dp_inverse, 'enumerate_splits': enumerate_splits,
         'submit': lambda s: Solution().countBalancedPermutations(s)}
edge = [("00", 1), ("11", 1), ("10", 0), ("999", 0), ("1122", 4), ("0000000", 1), ("123321", 36)]
for num, exp in examples + edge:
    assert brute(num) == exp, (num, brute(num))
    for name, f in impls.items():
        assert f(num) == exp, (name, num, f(num))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3343)
cnt_tests = 0
for _ in range(400):
    alphabet = '0123456789'[:rng.randint(1, 10)]
    num = ''.join(rng.choice(alphabet) for _ in range(rng.randint(2, 8)))
    exp = brute(num)
    for name, f in impls.items():
        assert f(num) == exp, (name, num)
    cnt_tests += 1
print('Khớp vét cạn trên', cnt_tests, 'chuỗi ngẫu nhiên.')

Khớp vét cạn trên 400 chuỗi ngẫu nhiên.


### 12.3. Ca lớn `n = 80`

In [12]:
import time
def random_even_sum(length):
    while True:
        q = ''.join(rng.choice('0123456789') for _ in range(length))
        if sum(map(int, q)) % 2 == 0:
            return q

cases = {
    'ngẫu nhiên 80 chữ số': random_even_sum(80),
    '40 số 9 + 40 số 1': '9' * 40 + '1' * 40,
    'mỗi chữ số 8 lần': ''.join(str(d) * 8 for d in range(10)),
    '79 chữ số (n lẻ)': random_even_sum(79),
}
for name, num in cases.items():
    t = time.time(); a = Solution().countBalancedPermutations(num); t1 = time.time() - t
    t = time.time(); b = memo_dfs(num); t2 = time.time() - t
    assert a == b and len(num) == (79 if 'lẻ' in name else 80)
    print(f"{name:22s}: {a:10d} | nộp {t1:.2f}s | memo {t2:.2f}s")

ngẫu nhiên 80 chữ số  :  886931643 | nộp 0.01s | memo 0.02s
40 số 9 + 40 số 1     :  432969423 | nộp 0.00s | memo 0.00s
mỗi chữ số 8 lần      :  284226843 | nộp 0.02s | memo 0.02s
79 chữ số (n lẻ)      :  411213545 | nộp 0.02s | memo 0.02s


Kiểm tra độc lập bằng số nguyên chính xác (cách 4) cho chuỗi dài nhưng ít loại chữ số: `"9"*40 + "1"*40` — nhóm chẵn cần `20` chữ số `9` và `20` chữ số `1` ⇒ đáp án `C(40,20)²`.

In [13]:
num = '9' * 40 + '1' * 40
assert Solution().countBalancedPermutations(num) == enumerate_splits(num) == comb(40, 20) ** 2 % MOD
print('"9"*40 + "1"*40: khớp C(40, 20)^2 mod p')

"9"*40 + "1"*40: khớp C(40, 20)^2 mod p


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm hoán vị theo **vị trí** (không khử trùng lặp)

In [14]:
def wrong_not_distinct(num):
    return sum(1 for p in permutations(num)
               if sum(int(p[i]) for i in range(0, len(p), 2)) == sum(int(p[i]) for i in range(1, len(p), 2)))

print('sai :', wrong_not_distinct("112"), '| đúng:', Solution().countBalancedPermutations("112"))

sai : 2 | đúng: 1


**Vì sao sai:** `"121"` xuất hiện hai lần (hoán đổi hai chữ số `1`) — đề đếm hoán vị **phân biệt**.

### Lỗi 2 — Không kiểm tra `S` lẻ

In [15]:
def wrong_no_parity(num):
    n = len(num)
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S = sum(map(int, num))
    half, nE = S // 2, (n + 1) // 2          # SAI: S lẻ thì S // 2 làm tròn xuống
    nO = n - nE
    dp = [[0] * (half + 1) for _ in range(nE + 1)]
    dp[0][0] = 1
    for d in range(10):
        c = cnt[d]
        nd = [[0] * (half + 1) for _ in range(nE + 1)]
        for j in range(nE + 1):
            for s in range(half + 1):
                if dp[j][s]:
                    for x in range(c + 1):
                        if j + x <= nE and s + x * d <= half:
                            nd[j + x][s + x * d] = (nd[j + x][s + x * d] + dp[j][s] * inv_fact[x] * inv_fact[c - x]) % MOD
        dp = nd
    return dp[nE][half] * fact[nE] % MOD * fact[nO] % MOD

print('sai :', wrong_no_parity("12"), '| đúng:', Solution().countBalancedPermutations("12"))

sai : 1 | đúng: 0


**Vì sao sai:** `"12"`: `half = 1`, DP tìm được nhóm chẵn `{1}`, nhưng nhóm lẻ `{2}` có tổng `2 ≠ 1`. DP chỉ ép tổng nhóm chẵn; nhóm lẻ bằng `S − half` chỉ đúng khi `S` chẵn.

### Lỗi 3 — Quên hệ số của nhóm lẻ (chỉ nhân `1/c!`)

In [16]:
def wrong_missing_odd_factor(num):
    n = len(num)
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S = sum(map(int, num))
    if S % 2:
        return 0
    half, nE = S // 2, (n + 1) // 2
    nO = n - nE
    dp = [[0] * (half + 1) for _ in range(nE + 1)]
    dp[0][0] = 1
    for d in range(10):
        c = cnt[d]
        nd = [[0] * (half + 1) for _ in range(nE + 1)]
        for j in range(nE + 1):
            for s in range(half + 1):
                if dp[j][s]:
                    for x in range(c + 1):
                        if j + x <= nE and s + x * d <= half:
                            nd[j + x][s + x * d] = (nd[j + x][s + x * d] + dp[j][s] * inv_fact[x]) % MOD   # SAI
        dp = nd
    return dp[nE][half] * fact[nE] % MOD * fact[nO] % MOD

bad = "1102"
print(repr(bad), 'sai :', wrong_missing_odd_factor(bad), '| đúng:', Solution().countBalancedPermutations(bad), '| vét cạn:', brute(bad))

'1102' sai : 6 | đúng: 4 | vét cạn: 4


**Vì sao sai:** nhóm lẻ cũng có thể chứa chữ số lặp — phải chia cả `(cnt − c)!`. Với `"1102"` (`nE = nO = 2`, `half = 2`): cách chia "chẵn `{0, 2}` | lẻ `{1, 1}`" chỉ cho `2!/1 · 2!/2! = 2` hoán vị (`"0121"`, `"2101"`), nhưng bản sai đếm `2 · 2 = 4` vì coi hai số `1` ở vị trí lẻ là khác nhau.

### Lỗi 4 — Chia giai thừa **đã lấy modulo** bằng phép chia nguyên

In [17]:
def wrong_integer_division(num):
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S, n = sum(map(int, num)), len(num)
    if S % 2:
        return 0
    nE = (n + 1) // 2
    total = 0
    for cs in product(*[range(c + 1) for c in cnt]):
        if sum(cs) == nE and sum(d * c for d, c in enumerate(cs)) == S // 2:
            a = fact[nE]
            for c in cs:
                a //= fact[c]                  # SAI: fact[] đã mod p, chia nguyên vô nghĩa
            b = fact[n - nE]
            for d in range(10):
                b //= fact[cnt[d] - cs[d]]
            total += a * b
    return total % MOD

bad = '9' * 20 + '1' * 20
print('sai :', wrong_integer_division(bad), '| đúng:', Solution().countBalancedPermutations(bad))

sai : 0 | đúng: 134779298


**Vì sao sai:** `fact[20] mod p` không còn chia hết cho `fact[10] mod p`. Trong modulo phải **nhân với nghịch đảo**. (Với `n` nhỏ, `n! < p` nên lỗi bị che giấu — chỉ lộ ra khi `n ≥ 13`.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm hoán vị **phân biệt** thỏa điều kiện trên hai nhóm vị trí" ⇒ chọn đa tập cho từng nhóm + hệ số đa thức.
- DP knapsack trên **giá trị chữ số** (chỉ 10 loại) thay vì trên từng ký tự — tránh đếm trùng.
- Trong modulo: thay `1/k!` bằng `inv_fact[k]`.

## 15. Biến thể của bài

1. Tổng chẵn **lớn hơn** tổng lẻ ⇒ cộng `dp[nE][s]` với `s > S − s`.
2. Chênh lệch đúng bằng `D` ⇒ `dp[nE][(S + D)/2]`.
3. Ba nhóm vị trí (theo `i mod 3`) — DP 2 chiều tổng.
4. Đếm số **xâu con** (không phải hoán vị) cân bằng — bài khác hẳn (tiền tố hiệu).

In [18]:
# Biến thể 1 và 2
def dp_table(num):
    n = len(num)
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S = sum(map(int, num)); nE = (n + 1) // 2
    dp = [[0] * (S + 1) for _ in range(nE + 1)]
    dp[0][0] = 1
    for d in range(10):
        c = cnt[d]
        nd = [[0] * (S + 1) for _ in range(nE + 1)]
        for j in range(nE + 1):
            for s in range(S + 1):
                if dp[j][s]:
                    for x in range(c + 1):
                        if j + x <= nE and s + x * d <= S:
                            nd[j + x][s + x * d] = (nd[j + x][s + x * d] + dp[j][s] * inv_fact[x] * inv_fact[c - x]) % MOD
        dp = nd
    return [v * fact[nE] % MOD * fact[n - nE] % MOD for v in dp[nE]], S

def even_greater(num):
    row, S = dp_table(num)
    return sum(row[s] for s in range(S + 1) if s > S - s) % MOD

def diff_exactly(num, D):
    row, S = dp_table(num)
    return row[(S + D) // 2] if (S + D) % 2 == 0 and 0 <= (S + D) // 2 <= S else 0

def brute_pred(num, pred):
    return sum(1 for p in set(permutations(num))
               if pred(sum(int(p[i]) for i in range(0, len(p), 2)), sum(int(p[i]) for i in range(1, len(p), 2))))

rng = random.Random(15)
for _ in range(200):
    num = ''.join(rng.choice('0123456') for _ in range(rng.randint(2, 7)))
    D = rng.randint(-6, 6)
    assert even_greater(num) == brute_pred(num, lambda e, o: e > o)
    assert diff_exactly(num, D) == brute_pred(num, lambda e, o: e - o == D)
print('Biến thể "chẵn > lẻ" và "chênh lệch đúng D" khớp vét cạn.')

Biến thể "chẵn > lẻ" và "chênh lệch đúng D" khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi hoán vị | `O(n! n)` | Oracle |
| 2 | Đệ quy + `C(e,c)·C(o,·)` | `O(10 · nE · half · cnt)` | Không cần nghịch đảo |
| 3 | DP lặp + `inv_fact` | `≈ 1.2·10^6` | **Nộp bài** |
| 4 | Liệt kê cách chia | `O(∏(cnt+1))` | Kiểm tra độc lập |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `"1111"`.

**Bài 2 (tính tay):** `"1230"`.

**Bài 3 (điền TODO):** đếm hoán vị phân biệt mà tổng vị trí chẵn **bằng 2 lần** tổng vị trí lẻ.

**Bài 4 (tự viết lại):** biến thể 3 — ba nhóm vị trí `i mod 3` có tổng bằng nhau.

**Bài 5 (mini challenge):** đếm hoán vị phân biệt cân bằng **không có số 0 đứng đầu**.

In [19]:
def double_todo(num):
    # TODO
    raise NotImplementedError

def three_groups_todo(num):
    # TODO
    raise NotImplementedError

def no_leading_zero_todo(num):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ một hoán vị phân biệt. (2) `1 + 1 = 1 + 1`. (3) `1`.

**Bài 2.** (1) `S = 6`, `half = 3`, `nE = nO = 2`. (2) Nhóm chẵn `{0, 3}` hoặc `{1, 2}`. (3) `2·2 + 2·2 = 8`.

**Bài 3.** (1) Tổng chẵn `e`, lẻ `o`, `e = 2o`, `e + o = S` ⇒ `S` chia hết cho 3, `e = 2S/3`. (2) Dùng `dp_table`. (3) Vét cạn.

**Bài 4.** (1) Ba nhóm kích thước `⌈(n − r)/3⌉`. (2) DP trạng thái `(j1, s1, j2, s2)`. (3) Cẩn thận độ phức tạp — chỉ cho `n` nhỏ.

**Bài 5.** (1) Lấy đáp án trừ số hoán vị cân bằng có `0` ở vị trí 0. (2) Cố định `0` ở vị trí 0: các chữ số còn lại (bỏ một `0`) chia vào `nE − 1` chỗ chẵn và `nO` chỗ lẻ. (3) Dùng lại đệ quy cách 2 với kích thước nhóm mới; số 0 ở vị trí 0 không đổi tổng.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().countBalancedPermutations("1111") == 1
assert Solution().countBalancedPermutations("1230") == 8 == brute("1230")

def double(num):
    row, S = dp_table(num)
    return row[2 * S // 3] if S % 3 == 0 else 0

rng = random.Random(3)
for _ in range(200):
    num = ''.join(rng.choice('0123456') for _ in range(rng.randint(2, 7)))
    assert double(num) == brute_pred(num, lambda e, o: e == 2 * o)
print('Bài 1: 1 | Bài 2: 8 | Bài 3 khớp vét cạn.')

Bài 1: 1 | Bài 2: 8 | Bài 3 khớp vét cạn.


In [21]:
# Bài 4, 5
def three_groups(num):
    n = len(num)
    sizes = [len(range(r, n, 3)) for r in range(3)]
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    S = sum(map(int, num))
    if S % 3:
        return 0
    T = S // 3
    @lru_cache(maxsize=None)
    def f(d, a, b, sa, sb):          # a, b = chỗ còn trống ở nhóm 0, 1; nhóm 2 suy ra
        if d == 10:
            return int(a == 0 and b == 0 and sa == 0 and sb == 0)
        res = 0
        used = sum(cnt[:d])
        cpos = sizes[2] - (used - (sizes[0] - a) - (sizes[1] - b))
        for x in range(min(cnt[d], a) + 1):
            for y in range(min(cnt[d] - x, b) + 1):
                z = cnt[d] - x - y
                if z > cpos or x * d > sa or y * d > sb:
                    continue
                res += comb(a, x) * comb(b, y) * comb(cpos, z) * f(d + 1, a - x, b - y, sa - x * d, sb - y * d)
        return res
    return f(0, sizes[0], sizes[1], T, T) % MOD

def no_leading_zero(num):
    total = Solution().countBalancedPermutations(num)
    if '0' not in num or len(num) == 1:
        return total
    rest = num.replace('0', '', 1)
    # cố định '0' ở vị trí 0: còn nE - 1 chỗ chẵn và nO chỗ lẻ cho các chữ số còn lại, tổng mỗi nhóm = S/2
    cnt = [0] * 10
    for ch in rest:
        cnt[int(ch)] += 1
    S = sum(map(int, rest))
    if S % 2:
        return total
    e_size, o_size = (len(num) + 1) // 2 - 1, len(num) // 2     # chỗ chẵn còn lại (trừ vị trí 0), chỗ lẻ
    @lru_cache(maxsize=None)
    def f(d, e, o, s):
        if d == 10:
            return int(e == 0 and o == 0 and s == 0)
        return sum(comb(e, c) * comb(o, cnt[d] - c) * f(d + 1, e - c, o - cnt[d] + c, s - c * d)
                   for c in range(cnt[d] + 1) if c <= e and cnt[d] - c <= o and c * d <= s)
    return (total - f(0, e_size, o_size, S // 2)) % MOD

def brute_three(num):
    return sum(1 for p in set(permutations(num))
               if len({sum(int(p[i]) for i in range(r, len(p), 3)) for r in range(3)}) == 1)

rng = random.Random(4)
for _ in range(150):
    num = ''.join(rng.choice('012345') for _ in range(rng.randint(2, 7)))
    assert three_groups(num) == brute_three(num) % MOD, num
    exp = sum(1 for p in set(permutations(num)) if p[0] != '0' and
              sum(int(p[i]) for i in range(0, len(p), 2)) == sum(int(p[i]) for i in range(1, len(p), 2)))
    assert no_leading_zero(num) == exp, num
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
S lẻ -> 0;  half = S//2;  nE = (n+1)//2;  nO = n//2
dp[0][0] = 1
for d in 0..9: for x in 0..cnt[d]:
    dp'[j+x][s+x*d] += dp[j][s] * inv_fact[x] * inv_fact[cnt[d]-x]
return dp[nE][half] * fact[nE] * fact[nO]
```

**Checklist:**

- [ ] Kiểm tra `S` lẻ ⇒ 0?
- [ ] `nE = ⌈n/2⌉` (chỉ số 0 là chẵn)?
- [ ] Chia cả `c!` và `(cnt − c)!` (hai nhóm đều khử lặp)?
- [ ] Dùng nghịch đảo modulo thay vì chia nguyên?
- [ ] Đã tạo biến `velunexorai` theo yêu cầu đề?